# 09 - Modelos de aprendizaje de máquina

Este notebook cubre los numerales 3 a 6 de la entrega final y el bono de aprendizaje profundo. Todo se entrena con MLlib sobre el cluster, a partir de la capa oro del notebook 07.

| Parte | Técnica | Pregunta que ayuda a responder |
|---|---|---|
| Supervisado | **Random Forest** (con regresión logística como referencia) | ¿Qué circunstancias de un choque elevan el riesgo de que haya heridos o muertos? |
| No supervisado | **K-Means** sobre los 55 barrios | ¿Qué barrios se parecen entre sí y cuáles deberían priorizarse juntos? |
| Bono | **Perceptrón multicapa** (red neuronal) | El mismo problema supervisado, para comparar |

**Por qué estas técnicas (numeral 3).**

- *Random Forest*: casi todas las variables del choque son categorías (causa, distrito, día) y su efecto no es lineal ni independiente: la hora pesa distinto según el día y el tipo de vehículo. Un bosque de árboles maneja eso sin que tengamos que definir las interacciones, admite pesos para compensar el desbalance de clases y entrega la importancia de cada variable, que es justo lo que pide el objetivo de negocio.
- *Regresión logística*: es el modelo más simple para este problema. Sirve de referencia: si el bosque no la supera, la complejidad extra no se justifica.
- *K-Means*: el objetivo pide priorizar zonas. Agrupar los 55 barrios por sus tasas permite proponer un mismo tipo de intervención para cada grupo en vez de 55 planes distintos. Los grupos se describen con sus centros, que son fáciles de explicar.

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, PercentFormatter
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, MultilayerPerceptronClassifier, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.stat import Correlation
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.sql import functions as F

import geo
from codigos import BANDERAS_VEHICULO
from config import GOLD, sesion
from estilo import BOROUGHS, CATEGORICA, SECUENCIAL, TINTA_SECUNDARIA, guardar, miles

spark = sesion("09_modelos")
SEMILLA = 42
UMBRAL_CORRELACION = 0.8     # por encima de esto (en valor absoluto) dos variables se consideran redundantes

def matriz_correlacion(df, columnas):
    """Correlación de Pearson entre columnas numéricas, calculada con MLlib."""
    vector = VectorAssembler(inputCols=columnas, outputCol="_v").transform(df.select(*[F.col(c).cast("double").alias(c) for c in columnas]))
    return pd.DataFrame(Correlation.corr(vector, "_v").head()[0].toArray(), index=columnas, columns=columnas)

def redundantes(correlacion, umbral=UMBRAL_CORRELACION):
    """Recorre las variables en orden de prioridad (el orden de las columnas). De cada pareja
    que supera el umbral se conserva la primera y se elimina la segunda."""
    eliminadas = {}
    columnas = list(correlacion.columns)
    for i, a in enumerate(columnas):
        if a in eliminadas:
            continue
        for b in columnas[i + 1:]:
            if b not in eliminadas and abs(correlacion.loc[a, b]) >= umbral:
                eliminadas[b] = {"se conserva": a, "correlación": round(correlacion.loc[a, b], 2)}
    return pd.DataFrame(eliminadas).T.rename_axis("variable eliminada")

def dibujar_correlacion(correlacion, titulo, nombre):
    cmap = LinearSegmentedColormap.from_list("divergente", [CATEGORICA[1], "#ffffff", CATEGORICA[0]])
    n = len(correlacion)
    fig, ax = plt.subplots(figsize=(0.62 * n + 2.6, 0.5 * n + 1.6))
    im = ax.imshow(correlacion.values, cmap=cmap, vmin=-1, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(correlacion.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n)); ax.set_yticklabels(correlacion.index, fontsize=8)
    for i in range(n):
        for j in range(n):
            v = correlacion.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if abs(v) > 0.6 else "#0b0b0b")
    ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    ax.set_title(titulo, pad=10)
    guardar(fig, nombre); plt.show()

# Parte A. Modelo supervisado: ¿el choque deja heridos o muertos?

## A.1 Datos y variable objetivo

Cada fila es un choque. La variable objetivo es `grave` (1 si hubo al menos un herido o un muerto). Se usan los choques que tienen distrito y hora, que son dos de las variables del modelo.

**Lo que no puede entrar al modelo.** Los conteos de heridos y muertos definen la variable objetivo; si entraran como variables, el modelo "adivinaría" con la respuesta en la mano (fuga de información). Por eso solo se usan circunstancias del choque: cuándo, dónde, por qué y con qué vehículos.

In [ ]:
choques = (spark.read.parquet(f"{GOLD}/choques_modelo")
           .filter(F.col("borough").isNotNull() & F.col("hora").isNotNull() & F.col("dia_semana").isNotNull()))

angulo = 2 * math.pi * F.col("hora") / 24
datos = (choques
    .withColumn("hora_sen", F.sin(angulo))      # la hora es circular: las 23 y las 0 están juntas
    .withColumn("hora_cos", F.cos(angulo))
    .withColumn("es_noche", ((F.col("hora") >= 20) | (F.col("hora") < 6)).cast("int"))
    .withColumn("n_vehiculos", F.col("n_vehiculos").cast("double"))
    .withColumn("grave", F.col("grave").cast("double")))

conteo = datos.groupBy("grave").count().toPandas().set_index("grave")["count"]
N, N_GRAVES, N_LEVES = int(conteo.sum()), int(conteo.loc[1.0]), int(conteo.loc[0.0])
print(f"Choques para el modelo: {N:,} | graves: {N_GRAVES:,} ({100 * N_GRAVES / N:.1f} %) | sin víctimas: {N_LEVES:,} ({100 * N_LEVES / N:.1f} %)")

## A.2 Preparación (numeral 4)

### a. Variables fuertemente correlacionadas

Variables numéricas candidatas, en orden de prioridad según el criterio de negocio. La hora se representa de varias maneras a propósito (el número, su seno y coseno, y si es de noche) para dejar que la correlación diga cuáles sobran. De cada pareja con correlación de 0,8 o más en valor absoluto se elimina la de menor prioridad.

In [ ]:
NUMERICAS_CANDIDATAS = ["hora_sen", "hora_cos", "n_vehiculos", *BANDERAS_VEHICULO, "mes", "fin_de_semana", "es_noche", "hora"]
CATEGORICAS = ["causa", "borough", "dia_semana"]

correlacion_a = matriz_correlacion(datos, NUMERICAS_CANDIDATAS)
dibujar_correlacion(correlacion_a, "Correlación entre las variables numéricas del choque", "09_01_correlacion_choques")
eliminadas_a = redundantes(correlacion_a)
correlacion_a.round(3).to_csv("../figuras/tabla_09_correlacion_choques.csv")
print("Eliminadas por correlación:" if len(eliminadas_a) else f"Ninguna pareja llega a {UMBRAL_CORRELACION} de correlación.")
eliminadas_a

### c. Variables finales según el criterio de negocio

Además de lo que elimine la correlación, se quitan dos variables por criterio propio:

- `hora` como número: trata las 23:00 y las 0:00 como extremos opuestos cuando en realidad son contiguas. El seno y el coseno conservan esa cercanía.
- `fin_de_semana`: es una función exacta del día de la semana, que ya entra como categoría. La correlación de Pearson no lo detecta porque el día es una categoría, no un número.

In [ ]:
POR_CRITERIO = ["hora", "fin_de_semana"]
NUMERICAS = [c for c in NUMERICAS_CANDIDATAS if c not in set(eliminadas_a.index) | set(POR_CRITERIO)]

DESCRIPCION = {
    "hora_sen": "hora del choque (componente seno)", "hora_cos": "hora del choque (componente coseno)",
    "es_noche": "1 si ocurrió entre las 20:00 y las 5:59", "n_vehiculos": "número de vehículos involucrados",
    "hay_moto": "1 si participó una motocicleta", "hay_bici": "1 si participó una bicicleta",
    "hay_micromovilidad": "1 si participó una bicicleta o patineta eléctrica", "hay_camion": "1 si participó un camión",
    "hay_bus": "1 si participó un bus", "hay_taxi": "1 si participó un taxi", "mes": "mes del año (1 a 12)",
    "causa": "causa del choque (12 grupos, más 'Otra causa' y 'Sin especificar')", "borough": "distrito (5)", "dia_semana": "día de la semana (7)",
}
variables_finales = pd.DataFrame([{"variable": v, "tipo": "numérica" if v in NUMERICAS else "categórica", "descripción": DESCRIPCION[v]}
                                  for v in NUMERICAS + CATEGORICAS])
variables_finales.to_csv("../figuras/tabla_09_variables_supervisado.csv", index=False)
print(f"Variables finales: {len(NUMERICAS)} numéricas y {len(CATEGORICAS)} categóricas")
variables_finales

### b. Normalización, codificación y partición

- **Partición**: 80 % de los choques para entrenar y 20 % para la prueba final, al azar y con semilla fija. La prueba no se toca hasta el final.
- **Normalización**: las numéricas se llevan a media 0 y desviación 1 con `StandardScaler`, ajustado solo con el conjunto de entrenamiento. La regresión logística y la red neuronal la necesitan; a los árboles no les afecta la escala.
- **Categorías**: se convierten en índices con `StringIndexer`. Los árboles usan los índices directamente; la regresión y la red los reciben como columnas de ceros y unos (`OneHotEncoder`).
- **Desbalance**: hay menos choques graves que sin víctimas. Cada choque recibe un peso inverso al tamaño de su clase, para que el modelo no gane simplemente prediciendo siempre "sin víctimas".

In [ ]:
datos = datos.withColumn("peso", F.when(F.col("grave") == 1, N / (2 * N_GRAVES)).otherwise(N / (2 * N_LEVES)))
entrenamiento, prueba = datos.randomSplit([0.8, 0.2], seed=SEMILLA)

indices = [f"{c}_idx" for c in CATEGORICAS]
binarias = [f"{c}_bin" for c in CATEGORICAS]
preparacion = Pipeline(stages=[
    StringIndexer(inputCols=CATEGORICAS, outputCols=indices, handleInvalid="keep"),
    OneHotEncoder(inputCols=indices, outputCols=binarias, handleInvalid="keep"),
    VectorAssembler(inputCols=NUMERICAS, outputCol="numericas"),
    StandardScaler(inputCol="numericas", outputCol="numericas_norm", withMean=True, withStd=True),
    VectorAssembler(inputCols=["numericas_norm", *binarias], outputCol="features"),            # regresión y red neuronal
    VectorAssembler(inputCols=[*NUMERICAS, *indices], outputCol="features_arbol"),              # árboles
]).fit(entrenamiento)

columnas = ["collision_id", "grave", "peso", "features", "features_arbol"]
entrenamiento_p = preparacion.transform(entrenamiento).select(*columnas).cache()
prueba_p = preparacion.transform(prueba).select(*columnas).cache()
N_ENTRADAS = entrenamiento_p.first()["features"].size
print(f"Entrenamiento: {entrenamiento_p.count():,} | Prueba: {prueba_p.count():,} | columnas de entrada tras codificar: {N_ENTRADAS}")

# Verificación: la proporción de graves debe ser casi igual en las dos partes
print("Graves en entrenamiento: {:.2f} % | en prueba: {:.2f} %".format(
    100 * entrenamiento_p.agg(F.avg("grave")).first()[0], 100 * prueba_p.agg(F.avg("grave")).first()[0]))

## A.3 Entrenamiento con distintos parámetros (numerales 5 y 6)

Para cada técnica se prueban varias combinaciones de parámetros. Cada combinación se entrena con el 80 % de una muestra del conjunto de entrenamiento y se mide con el 20 % restante (validación). La que obtiene mejor área bajo la curva ROC se vuelve a entrenar con todo el conjunto de entrenamiento. La búsqueda se hace sobre una muestra del 30 % para que el tiempo sea razonable en un cluster de 8 cores.

**Métricas.** La exactitud (porcentaje de aciertos) engaña cuando las clases están desbalanceadas, así que no es la métrica principal. Se usan:

- **AUC-ROC**: probabilidad de que el modelo le dé más riesgo a un choque grave que a uno sin víctimas. 0,5 es adivinar al azar y 1 es perfecto. No depende del umbral de decisión.
- **AUC-PR**: parecida, pero centrada en la clase que interesa (los graves). Su punto de comparación es la proporción de graves.
- **Precisión, sensibilidad y F1 de la clase "grave"**: de los que el modelo marca como graves, cuántos lo son; de los graves reales, cuántos detecta; y el balance de las dos.

In [ ]:
muestra = entrenamiento_p.sample(fraction=0.3, seed=SEMILLA).cache()
print(f"Muestra para la búsqueda de parámetros: {muestra.count():,} choques")
auc_roc = BinaryClassificationEvaluator(labelCol="grave", metricName="areaUnderROC")
auc_pr = BinaryClassificationEvaluator(labelCol="grave", metricName="areaUnderPR")
busquedas, modelos = [], {}

def buscar(nombre, estimador, rejilla, datos_busqueda, datos_finales):
    """Prueba cada combinación de la rejilla y reentrena la mejor con todos los datos de entrenamiento."""
    inicio = time.time()
    ajuste = TrainValidationSplit(estimator=estimador, estimatorParamMaps=rejilla, evaluator=auc_roc,
                                  trainRatio=0.8, seed=SEMILLA, parallelism=2).fit(datos_busqueda)
    for combinacion, metrica in zip(rejilla, ajuste.validationMetrics):
        busquedas.append({"técnica": nombre, "parámetros": ", ".join(f"{p.name}={v}" for p, v in combinacion.items()),
                          "AUC-ROC validación": round(metrica, 4)})
    mejor = rejilla[int(np.argmax(ajuste.validationMetrics))]
    modelos[nombre] = estimador.copy(mejor).fit(datos_finales)
    print(f"{nombre}: mejor combinación -> {', '.join(f'{p.name}={v}' for p, v in mejor.items())} "
          f"(AUC-ROC validación {max(ajuste.validationMetrics):.4f}; {time.time() - inicio:.0f} s)")

In [ ]:
logistica = LogisticRegression(featuresCol="features", labelCol="grave", weightCol="peso", maxIter=100)
buscar("Regresión logística", logistica,
       ParamGridBuilder().addGrid(logistica.regParam, [0.0, 0.01, 0.1]).addGrid(logistica.elasticNetParam, [0.0, 0.5]).build(),
       muestra, entrenamiento_p)

In [ ]:
bosque = RandomForestClassifier(featuresCol="features_arbol", labelCol="grave", weightCol="peso", seed=SEMILLA,
                                maxBins=32, subsamplingRate=0.7)
buscar("Random Forest", bosque,
       ParamGridBuilder().addGrid(bosque.numTrees, [40, 80]).addGrid(bosque.maxDepth, [6, 10, 14]).build(),
       muestra, entrenamiento_p)

### Bono: red neuronal (perceptrón multicapa)

El `MultilayerPerceptronClassifier` de MLlib es una red neuronal de capas completamente conectadas. Recibe las mismas variables normalizadas y codificadas que la regresión logística. La capa de entrada tiene una neurona por columna, la de salida dos (grave o no) y en medio se prueban una y dos capas ocultas.

Esta clase no admite pesos por fila. Para compensar el desbalance se entrena con una muestra **balanceada**: todos los choques graves y una muestra al azar de igual tamaño de los que no lo son. La evaluación se hace sobre la prueba original, sin balancear, igual que los otros modelos.

In [ ]:
fracciones = {0.0: min(1.0, N_GRAVES / N_LEVES), 1.0: min(1.0, N_LEVES / N_GRAVES)}
entrenamiento_bal = entrenamiento_p.sampleBy("grave", fractions=fracciones, seed=SEMILLA).cache()
muestra_bal = muestra.sampleBy("grave", fractions=fracciones, seed=SEMILLA).cache()
print(f"Entrenamiento balanceado: {entrenamiento_bal.count():,} choques ({100 * entrenamiento_bal.agg(F.avg('grave')).first()[0]:.1f} % graves)")

red = MultilayerPerceptronClassifier(featuresCol="features", labelCol="grave", seed=SEMILLA, solver="l-bfgs", blockSize=256)
buscar("Red neuronal (MLP)", red,
       ParamGridBuilder().addGrid(red.layers, [[N_ENTRADAS, 16, 2], [N_ENTRADAS, 32, 16, 2]]).addGrid(red.maxIter, [60, 120]).build(),
       muestra_bal, entrenamiento_bal)

In [ ]:
busquedas_df = pd.DataFrame(busquedas)
busquedas_df.to_csv("../figuras/tabla_09_busqueda_parametros.csv", index=False)
pd.set_option("display.max_colwidth", 120)
busquedas_df

## A.4 Evaluación en el conjunto de prueba

Los tres modelos se miden con el 20 % de choques que ninguno vio durante el entrenamiento. Como referencia se incluye lo que obtendría alguien que respondiera siempre "sin víctimas".

In [ ]:
def evaluar(nombre, modelo):
    pred = modelo.transform(prueba_p).select("grave", "prediction", "rawPrediction", vector_to_array(F.col("probability"))[1].alias("prob")).cache()
    m = {(int(f["grave"]), int(f["prediction"])): f["count"] for f in pred.groupBy("grave", "prediction").count().collect()}
    vp, fp, fn, vn = m.get((1, 1), 0), m.get((0, 1), 0), m.get((1, 0), 0), m.get((0, 0), 0)
    precision, sensibilidad = vp / max(vp + fp, 1), vp / max(vp + fn, 1)
    fila = {
        "modelo": nombre,
        "AUC-ROC": auc_roc.evaluate(pred), "AUC-PR": auc_pr.evaluate(pred),
        "precisión (grave)": precision, "sensibilidad (grave)": sensibilidad,
        "F1 (grave)": 2 * precision * sensibilidad / max(precision + sensibilidad, 1e-9),
        "especificidad": vn / max(vn + fp, 1), "exactitud": (vp + vn) / (vp + fp + fn + vn),
        "VP": vp, "FP": fp, "FN": fn, "VN": vn,
    }
    curva = pred.select("grave", "prob").toPandas().sort_values("prob", ascending=False)
    pred.unpersist()
    return fila, curva

resultados, curvas = [], {}
for nombre, modelo in modelos.items():
    fila, curvas[nombre] = evaluar(nombre, modelo)
    resultados.append(fila)

proporcion_graves = prueba_p.agg(F.avg("grave")).first()[0]
resultados.append({"modelo": "Referencia: siempre 'sin víctimas'", "AUC-ROC": 0.5, "AUC-PR": proporcion_graves,
                   "precisión (grave)": 0.0, "sensibilidad (grave)": 0.0, "F1 (grave)": 0.0, "especificidad": 1.0,
                   "exactitud": 1 - proporcion_graves})
evaluacion = pd.DataFrame(resultados).set_index("modelo")
evaluacion.round(4).to_csv("../figuras/tabla_09_evaluacion_supervisado.csv")
evaluacion[["AUC-ROC", "AUC-PR", "precisión (grave)", "sensibilidad (grave)", "F1 (grave)", "especificidad", "exactitud"]].round(3)

In [ ]:
COLOR_MODELO = dict(zip(modelos, [CATEGORICA[0], CATEGORICA[1], CATEGORICA[2]]))
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
for nombre, curva in curvas.items():
    positivos = np.cumsum(curva["grave"].values); negativos = np.cumsum(1 - curva["grave"].values)
    paso = max(len(curva) // 400, 1)
    ax.plot(negativos[::paso] / negativos[-1], positivos[::paso] / positivos[-1], color=COLOR_MODELO[nombre],
            label=f"{nombre} (AUC {evaluacion.loc[nombre, 'AUC-ROC']:.3f})".replace(".", ","))
ax.plot([0, 1], [0, 1], color=TINTA_SECUNDARIA, linestyle=":", linewidth=1, label="Al azar (AUC 0,500)")
ax.set_xlabel("falsos positivos (choques sin víctimas marcados como graves)"); ax.set_ylabel("graves detectados (sensibilidad)")
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.legend(fontsize=8, loc="lower right"); ax.set_title("Curva ROC en el conjunto de prueba", pad=10)

metricas = ["AUC-ROC", "AUC-PR", "F1 (grave)", "sensibilidad (grave)", "precisión (grave)"]
ancho = 0.25
for i, nombre in enumerate(modelos):
    valores = evaluacion.loc[nombre, metricas].astype(float).values
    ax2.bar(np.arange(len(metricas)) + (i - 1) * ancho, valores, width=ancho, color=COLOR_MODELO[nombre], label=nombre)
    for x, v in zip(np.arange(len(metricas)) + (i - 1) * ancho, valores):
        ax2.text(x, v, f"{v:.2f}".replace(".", ","), ha="center", va="bottom", fontsize=7, color=TINTA_SECUNDARIA)
ax2.set_xticks(range(len(metricas))); ax2.set_xticklabels([m.replace(" (grave)", "\n(grave)") for m in metricas], fontsize=8)
ax2.set_ylim(0, 1); ax2.grid(axis="x", visible=False); ax2.legend(fontsize=8, loc="upper right")
ax2.set_title("Métricas en el conjunto de prueba", pad=10)
fig.tight_layout()
guardar(fig, "09_02_evaluacion_supervisado"); plt.show()

### Matriz de confusión e importancia de las variables

La matriz de confusión del Random Forest muestra en qué se equivoca. La importancia de las variables dice cuánto aporta cada una a separar los choques graves de los demás; es la parte del modelo que responde al objetivo de negocio.

In [ ]:
f = evaluacion.loc["Random Forest"]
confusion = pd.DataFrame([[f["VN"], f["FP"]], [f["FN"], f["VP"]]], index=["Real: sin víctimas", "Real: grave"],
                         columns=["Predicho: sin víctimas", "Predicho: grave"]).astype(int)
confusion.to_csv("../figuras/tabla_09_confusion_random_forest.csv")

NOMBRE_VARIABLE = {**{v: v for v in NUMERICAS}, **{f"{c}_idx": c for c in CATEGORICAS}}
importancia = pd.Series(modelos["Random Forest"].featureImportances.toArray(),
                        index=[NOMBRE_VARIABLE[c] for c in [*NUMERICAS, *indices]]).sort_values()
importancia.sort_values(ascending=False).round(4).to_csv("../figuras/tabla_09_importancia_variables.csv", header=["importancia"])

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={"width_ratios": [1, 1.4]})
im = ax.imshow(confusion.values, cmap=LinearSegmentedColormap.from_list("azules", SECUENCIAL))
for i in range(2):
    for j in range(2):
        v = confusion.values[i, j]
        ax.text(j, i, f"{miles(v)}\n{100 * v / confusion.values[i].sum():.1f} % de la fila", ha="center", va="center", fontsize=9,
                color="white" if v > confusion.values.max() / 2 else "#0b0b0b")
ax.set_xticks([0, 1]); ax.set_xticklabels(["sin víctimas", "grave"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["sin víctimas", "grave"])
ax.set_xlabel("lo que predijo el modelo"); ax.set_ylabel("lo que pasó"); ax.grid(False)
ax.set_title("Matriz de confusión del Random Forest", pad=10)
ax2.barh(importancia.index, importancia.values, color=CATEGORICA[0], height=0.65)
for y, v in zip(importancia.index, importancia.values):
    ax2.text(v, y, f" {100 * v:.1f} %", va="center", fontsize=8, color=TINTA_SECUNDARIA)
ax2.xaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax2.grid(axis="y", visible=False); ax2.margins(x=0.12)
ax2.set_title("Importancia de cada variable en el Random Forest", pad=10)
fig.tight_layout()
guardar(fig, "09_03_confusion_e_importancia"); plt.show()
confusion

In [ ]:
mejor = evaluacion.drop(index="Referencia: siempre 'sin víctimas'")["AUC-ROC"].astype(float).idxmax()
print(f"Resultado: el modelo con mayor AUC-ROC en la prueba es {mejor} ({evaluacion.loc[mejor, 'AUC-ROC']:.3f}).")
for nombre in modelos:
    e = evaluacion.loc[nombre]
    print(f"  {nombre}: AUC-ROC {e['AUC-ROC']:.3f}, AUC-PR {e['AUC-PR']:.3f} (referencia {proporcion_graves:.3f}), "
          f"detecta el {100 * e['sensibilidad (grave)']:.1f} % de los graves con una precisión del {100 * e['precisión (grave)']:.1f} %.")
orden_importancia = importancia.sort_values(ascending=False)
print("Variables más importantes del Random Forest:", ", ".join(f"{v} ({100 * x:.1f} %)" for v, x in orden_importancia.head(5).items()))
print(f"Las cinco primeras suman el {100 * orden_importancia.head(5).sum():.1f} % de la importancia.")

**Cómo leer estos resultados.** Un AUC-ROC cercano a 0,5 significaría que las circunstancias no dicen nada sobre la gravedad; cercano a 1, que la determinan por completo. El valor que se obtenga hay que leerlo sabiendo que el modelo no conoce cosas que pesan mucho en la gravedad de un choque: la velocidad real, si había peatones, el uso del cinturón o el estado de la vía. Aun así, el modelo sirve para lo que pide el objetivo: ordenar qué circunstancias se asocian con más riesgo. No sirve para predecir choques antes de que ocurran, porque la causa y los vehículos solo se conocen después.

Las importancias dicen cuánto usa el modelo cada variable, no en qué dirección empuja. La dirección está en la pregunta 5 del notebook 08 (porcentaje de graves por causa y por tipo de vehículo).

In [ ]:
for df in (entrenamiento_p, prueba_p, muestra, entrenamiento_bal, muestra_bal):
    df.unpersist()

# Parte B. Modelo no supervisado: grupos de barrios

## B.1 Datos

Una fila por barrio (55), con sus tasas anuales por 100.000 habitantes y sus porcentajes, tomadas de la tabla `barrios` de la capa oro.

**Transformación previa.** Las tasas están muy concentradas: unos pocos barrios centrales, a donde llega mucha gente que no vive allí, tienen tasas varias veces mayores que el resto. K-Means usa distancias, así que esos barrios dominarían el resultado y quedarían solos en un grupo. Para evitarlo se aplica el logaritmo a las tasas antes de normalizar. Los porcentajes no lo necesitan.

In [ ]:
barrios = spark.read.parquet(f"{GOLD}/barrios").toPandas()
boroughs = spark.read.parquet(f"{GOLD}/boroughs").toPandas().set_index("borough")
pumas = geo.cargar_pumas(spark)

# En orden de prioridad según el criterio de negocio: primero los dos indicadores del caso
CANDIDATAS_B = ["tasa_arrestos", "tasa_choques_graves", "pct_choques_graves", "pct_felonias", "tasa_arrestos_transito",
                "tasa_felonias", "tasa_delitos_menores", "tasa_choques", "tasa_heridos", "tasa_arrestos_embriaguez"]
transformadas = barrios[["puma"]].copy()
for c in CANDIDATAS_B:
    transformadas[c] = np.log(barrios[c]) if c.startswith("tasa_") else barrios[c]
assert np.isfinite(transformadas[CANDIDATAS_B]).all().all(), "Hay tasas en cero: el logaritmo no se puede aplicar"

asimetria = pd.DataFrame({"asimetría original": barrios[CANDIDATAS_B].skew().round(2), "asimetría tras el logaritmo": transformadas[CANDIDATAS_B].skew().round(2)})
asimetria

## B.2 Preparación (numeral 4)

### a. Variables fuertemente correlacionadas

Varias tasas miden casi lo mismo: la de felonías es parte de la de arrestos, y la de heridos sigue a la de choques graves. Si entraran todas, K-Means contaría el mismo fenómeno varias veces. Se aplica la misma regla que en el modelo supervisado.

In [ ]:
barrios_sdf = spark.createDataFrame(transformadas).coalesce(1)   # 55 filas: una sola partición
correlacion_b = matriz_correlacion(barrios_sdf, CANDIDATAS_B)
dibujar_correlacion(correlacion_b, "Correlación entre los indicadores de los 55 barrios", "09_04_correlacion_barrios")
eliminadas_b = redundantes(correlacion_b)
correlacion_b.round(3).to_csv("../figuras/tabla_09_correlacion_barrios.csv")
VARIABLES_B = [c for c in CANDIDATAS_B if c not in set(eliminadas_b.index)]
print("Variables finales para K-Means:", VARIABLES_B)
eliminadas_b

### b y c. Normalización y variables finales

Las variables que quedan se normalizan (media 0, desviación 1) para que todas pesen lo mismo en la distancia, sin importar su unidad.

La pobreza y la educación no entran como variables porque solo existen por distrito: todos los barrios de un mismo distrito tendrían el mismo valor y el algoritmo terminaría agrupando por distrito. Se usan después, para describir los grupos.

In [ ]:
preparados = Pipeline(stages=[
    VectorAssembler(inputCols=VARIABLES_B, outputCol="vector"),
    StandardScaler(inputCol="vector", outputCol="features", withMean=True, withStd=True),
]).fit(barrios_sdf).transform(barrios_sdf).select("puma", "features").cache()

## B.3 K-Means con distintos parámetros (numerales 5 y 6)

Se prueba de 2 a 8 grupos, cada uno con tres semillas distintas, porque K-Means parte de centros al azar y puede dar resultados diferentes. Dos métricas:

- **Silueta** (de -1 a 1): compara qué tan cerca está cada barrio de su grupo frente al grupo vecino. Más alto es mejor.
- **Inercia**: suma de las distancias al cuadrado de cada barrio al centro de su grupo. Siempre baja al aumentar los grupos; lo que se busca es el "codo", el punto donde deja de bajar rápido.

In [ ]:
silueta = ClusteringEvaluator(featuresCol="features", metricName="silhouette", distanceMeasure="squaredEuclidean")
SEMILLAS = [1, 7, 42]
pruebas = []
for k in range(2, 9):
    for semilla in SEMILLAS:
        modelo_k = KMeans(featuresCol="features", k=k, seed=semilla, maxIter=50).fit(preparados)
        asignados = modelo_k.transform(preparados)
        tamanos = asignados.groupBy("prediction").count().toPandas()["count"]
        pruebas.append({"k": k, "semilla": semilla, "silueta": silueta.evaluate(asignados), "inercia": modelo_k.summary.trainingCost,
                        "grupo_mas_pequeno": int(tamanos.min())})
pruebas = pd.DataFrame(pruebas)
resumen_k = pruebas.groupby("k").agg(silueta_media=("silueta", "mean"), silueta_min=("silueta", "min"), silueta_max=("silueta", "max"),
                                     inercia_media=("inercia", "mean"), grupo_mas_pequeno=("grupo_mas_pequeno", "min")).round(3)
pruebas.round(4).to_csv("../figuras/tabla_09_kmeans_pruebas.csv", index=False)
resumen_k.to_csv("../figuras/tabla_09_kmeans_resumen.csv")

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.8))
ax.plot(resumen_k.index, resumen_k["silueta_media"], marker="o", color=CATEGORICA[0])
ax.fill_between(resumen_k.index, resumen_k["silueta_min"], resumen_k["silueta_max"], color=CATEGORICA[0], alpha=0.15)
ax.set_xlabel("número de grupos (k)"); ax.set_title("Silueta (media y rango de 3 semillas)", pad=10)
ax2.plot(resumen_k.index, resumen_k["inercia_media"], marker="o", color=CATEGORICA[1])
ax2.set_xlabel("número de grupos (k)"); ax2.set_title("Inercia (método del codo)", pad=10)
fig.tight_layout()
guardar(fig, "09_05_kmeans_silueta_codo"); plt.show()
resumen_k

**Elección de k.** Se toma el k con mayor silueta media entre 3 y 6. Se descarta k = 2 aunque tenga buena silueta, porque dos grupos ("alto" y "bajo") no alcanzan para diferenciar intervenciones; y más de 6 deja grupos de muy pocos barrios, difíciles de sostener con solo 55 observaciones (la columna `grupo_mas_pequeno` muestra cuándo aparece un grupo de un solo barrio). Dentro del k elegido se usa la semilla con mejor silueta.

In [ ]:
K = int(resumen_k.loc[3:6, "silueta_media"].idxmax())
semilla_final = int(pruebas[pruebas["k"] == K].sort_values("silueta", ascending=False).iloc[0]["semilla"])
kmeans = KMeans(featuresCol="features", k=K, seed=semilla_final, maxIter=50).fit(preparados)
asignacion = kmeans.transform(preparados).select("puma", "prediction").toPandas()
silueta_final = silueta.evaluate(kmeans.transform(preparados))
print(f"k = {K} (semilla {semilla_final}) | silueta {silueta_final:.3f} | inercia {kmeans.summary.trainingCost:.1f}")

# Los grupos se renumeran de mayor a menor tasa de arrestos para que el número signifique algo
barrios = barrios.merge(asignacion, on="puma")
orden_grupos = barrios.groupby("prediction")["tasa_arrestos"].mean().sort_values(ascending=False).index
barrios["grupo"] = barrios["prediction"].map({original: f"Grupo {i + 1}" for i, original in enumerate(orden_grupos)})
GRUPOS = [f"Grupo {i + 1}" for i in range(K)]
COLOR_GRUPO = dict(zip(GRUPOS, ["#e34948", CATEGORICA[3], CATEGORICA[0], CATEGORICA[2], CATEGORICA[6], CATEGORICA[4]]))

## B.4 Qué caracteriza a cada grupo

El perfil muestra el promedio de cada indicador en sus unidades originales. Las columnas de pobreza y educación son las del distrito de cada barrio (promedio ponderado por población) y solo sirven de contexto: no participaron en la agrupación.

In [ ]:
barrios["pobreza_distrito"] = barrios["borough"].map(boroughs["pobreza_pct"])
barrios["pregrado_distrito"] = barrios["borough"].map(boroughs["pregrado_pct"])

def ponderado(grupo, columna):
    return np.average(grupo[columna], weights=grupo["poblacion_2018"])

perfil = barrios.groupby("grupo").apply(lambda g: pd.Series({
    "barrios": len(g),
    "población": g["poblacion_2018"].sum(),
    **{c: g[c].mean() for c in ["tasa_arrestos", "tasa_felonias", "tasa_choques_graves", "pct_choques_graves", "pct_felonias", "tasa_arrestos_transito"]},
    "pobreza del distrito %": ponderado(g, "pobreza_distrito"),
    "pregrado del distrito %": ponderado(g, "pregrado_distrito"),
    "distritos": ", ".join(f"{b} ({n})" for b, n in g["borough"].value_counts().items()),
})).reindex(GRUPOS)
for c in perfil.columns.drop("distritos"):      # apply devuelve todo como texto; se vuelve a número
    perfil[c] = pd.to_numeric(perfil[c])
perfil.to_csv("../figuras/tabla_09_kmeans_perfil.csv")
barrios[["puma", "barrio", "borough", "grupo", "poblacion_2018", "tasa_arrestos", "tasa_choques_graves", "pct_choques_graves", "pct_felonias"]] \
    .sort_values(["grupo", "tasa_arrestos"], ascending=[True, False]).round(1).to_csv("../figuras/tabla_09_kmeans_barrios.csv", index=False)
perfil.round(1)

In [ ]:
fig, (ax, ax_mapa) = plt.subplots(1, 2, figsize=(12.5, 5.6), gridspec_kw={"width_ratios": [1.15, 1]})
for grupo in GRUPOS:
    g = barrios[barrios["grupo"] == grupo]
    ax.scatter(g["tasa_arrestos"], g["tasa_choques_graves"], color=COLOR_GRUPO[grupo], s=45, label=f"{grupo} ({len(g)} barrios)", zorder=3)
ax.set_xscale("log"); ax.set_yscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(miles)); ax.yaxis.set_major_formatter(FuncFormatter(miles))
ax.set_xlabel("arrestos por 100.000 hab. al año (escala logarítmica)"); ax.set_ylabel("choques graves por 100.000 hab. al año (escala logarítmica)")
ax.legend(fontsize=8); ax.set_title(f"Los {K} grupos de barrios según K-Means", pad=10)
geo.dibujar_pumas(ax_mapa, pumas, {p: COLOR_GRUPO[g] for p, g in zip(barrios["puma"], barrios["grupo"])})
ax_mapa.legend(handles=[Patch(color=COLOR_GRUPO[g], label=g) for g in GRUPOS], fontsize=8, loc="upper left")
ax_mapa.set_title("Dónde están", pad=10)
fig.tight_layout()
guardar(fig, "09_06_kmeans_grupos"); plt.show()

In [ ]:
media_ciudad = barrios[["tasa_arrestos", "tasa_choques_graves"]].mean()
print(f"Resultado: K-Means con k = {K} separa los 55 barrios con una silueta de {silueta_final:.2f}.")
for grupo in GRUPOS:
    p, g = perfil.loc[grupo], barrios[barrios["grupo"] == grupo]
    print(f"\n{grupo}: {int(p['barrios'])} barrios, {miles(p['población'])} habitantes ({p['distritos']}).")
    print(f"   Arrestos {miles(p['tasa_arrestos'])} por 100.000 ({p['tasa_arrestos'] / media_ciudad['tasa_arrestos']:.1f} veces el promedio de los barrios); "
          f"choques graves {miles(p['tasa_choques_graves'])} ({p['tasa_choques_graves'] / media_ciudad['tasa_choques_graves']:.1f} veces); "
          f"{p['pct_choques_graves']:.1f} % de sus choques son graves.")
    print("   Barrios: " + "; ".join(g.sort_values("tasa_arrestos", ascending=False)["barrio"].head(8)) + ("; ..." if len(g) > 8 else ""))

**Cómo leer estos resultados.** La silueta va de -1 a 1. Con 55 barrios y variables continuas no se esperan grupos perfectamente separados: los barrios forman un continuo y K-Means lo corta en tramos. Lo que importa para el negocio es que cada grupo tenga un perfil distinto y explicable, porque a cada uno le corresponde un tipo de intervención.

Limitaciones: los grupos dependen de las variables que se eligieron y de la población de residentes que se usa como denominador; un barrio muy cerca del límite entre dos grupos puede cambiar de grupo con otra semilla.

## Conclusiones

- **Supervisado.** Se entrenaron una regresión logística, un Random Forest y una red neuronal para anticipar si un choque deja víctimas, con varias combinaciones de parámetros cada uno y una prueba final sobre choques que ninguno había visto. Las métricas y la importancia de las variables quedan arriba, con las cifras de esta ejecución.
- **No supervisado.** K-Means agrupó los 55 barrios según sus tasas de arrestos y de choques; el número de grupos se eligió con la silueta y el método del codo.
- **Preparación.** En ambos casos se midió la correlación entre variables, se eliminaron las redundantes con un umbral de 0,8, se normalizó con `StandardScaler` y se justificó la selección final.
- **Bono.** El perceptrón multicapa de MLlib se entrenó sobre el mismo problema supervisado y se evaluó con las mismas métricas.
- Las tablas y figuras quedan en `figuras/` con el prefijo `09_`.